In [0]:
%pip install databricks-sdk>=0.89.0 "psycopg[binary]"
%restart_python

In [0]:
import time
import psycopg
from databricks.sdk import WorkspaceClient

# ---- from the Connect dialog (project > production branch > Connect) ----
PGHOST     = "ep-lingering-scene-d8pe7h97.database.us-east-2.cloud.databricks.com"
PGUSER     = "dais2026hackathon@mitimco.mit.edu"   # still need this — run the snippet below
PGDATABASE = "databricks_postgres"
ENDPOINT   = "projects/dais-postgres-db/branches/production/endpoints/ep-lingering-scene-d8pe7h97"
# -------------------------------------------------------------------------

time.sleep(2)  # Allow kernel to stabilize after restart

w = WorkspaceClient()                                  # auths as you (the notebook user)
cred = w.postgres.generate_database_credential(endpoint=ENDPOINT)   # Autoscaling API

conn = psycopg.connect(
    host=PGHOST, dbname=PGDATABASE, user=PGUSER,
    password=cred.token, sslmode="require",
)
with conn.cursor() as cur:
    cur.execute("select current_user, current_database()")
    print(cur.fetchone())

In [0]:
import numpy as np, pandas as pd

FAC_DDL = """
CREATE TABLE app.facilities (
  facility_id        text PRIMARY KEY,
  name               text,
  state              text,
  city               text,
  latitude           double precision,
  longitude          double precision,
  coord_source       text,
  postcode           text,
  specialties        text,
  description        text,
  capability         text,
  procedure          text,
  equipment          text,
  number_doctors     text,
  capacity           text,
  year_established   text,
  source_urls        text,
  pincode_district   text,
  pincode_state      text,
  completeness_score double precision,
  geo_status         text,
  pmjay_match        text
)
"""
PIN_DDL = """
CREATE TABLE app.pincodes (
  pincode    text PRIMARY KEY,
  district   text,
  state_name text,
  rep_lat    double precision,
  rep_lon    double precision,
  n_offices  integer
)
"""

FAC_COLS = ["facility_id","name","state","city","latitude","longitude","coord_source","postcode",
            "specialties","description","capability","procedure","equipment","number_doctors",
            "capacity","year_established","source_urls","pincode_district","pincode_state",
            "completeness_score","geo_status","pmjay_match"]
PIN_COLS = ["pincode","district","state_name","rep_lat","rep_lon","n_offices"]

def py(v):
    if v is None: return None
    if isinstance(v, float) and pd.isna(v): return None
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.floating): return None if np.isnan(v) else float(v)
    return v

def load(pg_table, src_table, ddl, cols):
    pdf = spark.table(src_table).toPandas()[cols]
    with conn.cursor() as cur:
        cur.execute("CREATE SCHEMA IF NOT EXISTS app")
        cur.execute(f"DROP TABLE IF EXISTS app.{pg_table}")
        cur.execute(ddl)
        with cur.copy(f"COPY app.{pg_table} ({','.join(cols)}) FROM STDIN") as cp:
            for r in pdf.itertuples(index=False, name=None):
                cp.write_row(tuple(py(v) for v in r))
    conn.commit()
    print(f"app.{pg_table}: {len(pdf)} rows loaded")

load("facilities", "workspace.bharosacare.facilities_curated", FAC_DDL, FAC_COLS)
load("pincodes",   "workspace.bharosacare.pincodes_curated",   PIN_DDL, PIN_COLS)

In [0]:
with conn.cursor() as cur:
    for t in ["facilities", "pincodes"]:
        cur.execute(f"select count(*) from app.{t}")
        print(f"app.{t}:", cur.fetchone()[0], "rows")
    cur.execute("""select column_name, data_type from information_schema.columns
                   where table_schema='app' and table_name='facilities' order by ordinal_position""")
    print("--- app.facilities columns ---")
    for c in cur.fetchall(): print(" ", c[0], c[1])
    cur.execute("select facility_id, name, latitude, longitude, completeness_score from app.facilities limit 5")
    print("--- sample ---")
    for r in cur.fetchall(): print(" ", r)
conn.close()